# Predicting next-month arrears
Question: can we predict `arrears_next_month` from account information known in the billing month?

Design decisions (following checklist.md):
- **Leakage:** `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time. It is dropped. `meter_ref` (identifier) and `billing_month` (used only for splitting) are not features.
- **Test set:** the model is meant for future rows and accounts repeat across rows, so the test set is the latest months *and* only whole held-out accounts. Those accounts never appear in training.
- **CV:** inside training rows only, forward-in-time folds where validation accounts are also disjoint from the fold's training accounts.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.base import clone
df=pd.read_csv('data.csv',parse_dates=['billing_month'])
TARGET='arrears_next_month'
LEAKY=['final_notice_sent']; IDS=['meter_ref']
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']
print(df.shape, df[TARGET].mean())

## 1. Test set, set aside before any modelling
Test accounts: random 20% of accounts. Test rows: those accounts in the last 6 billing months (2025-07 onward). Training rows: all other accounts, months up to 2025-05 (2025-06 is a one-month gap).

In [ ]:
rng=np.random.RandomState(0)
accts=df.meter_ref.unique()
test_accts=set(rng.choice(accts,size=int(0.2*len(accts)),replace=False))
CUT=pd.Timestamp('2025-07-01'); GAP=pd.Timestamp('2025-06-01')
is_test=df.meter_ref.isin(test_accts)&(df.billing_month>=CUT)
is_train=~df.meter_ref.isin(test_accts)&(df.billing_month<GAP)
test=df[is_test].copy(); train=df[is_train].sort_values('billing_month').reset_index(drop=True)
assert not set(test.meter_ref)&set(train.meter_ref)
print(len(train),len(test),train[TARGET].mean(),test[TARGET].mean())

## 2. Candidate models (named before scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: standardised numeric + one-hot tariff, logistic regression (C in 0.1, 1).
3. `hgb`: histogram gradient boosting, small grid over depth/learning rate (chosen in CV only).

Selection metric: CV average precision (PR-AUC; ~12% positives). Threshold: maximises F1 on pooled out-of-fold predictions of the chosen model.

In [ ]:
num=['occupants','kwh','bill_eur','late_payments_12m']
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),['tariff'])])
CANDS={'baseline':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1.0]:
    CANDS[f'logreg_C{C_}']=Pipeline([('p',pre(True)),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in [2,3]:
    for lr in [0.05,0.1]:
        CANDS[f'hgb_d{d}_lr{lr}']=Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=150,l2_regularization=1.0,random_state=0))])
print(list(CANDS))

## 3. Cross-validation on training rows
Expanding window; each fold validates on one 3-month block using one of four disjoint account groups, and trains on earlier months of the other accounts. Preprocessing lives in the pipeline, so it is refit per fold.

In [ ]:
tr_accts=train.meter_ref.unique()
grp=pd.Series(np.random.RandomState(1).randint(0,4,len(tr_accts)),index=tr_accts)
train['g']=train.meter_ref.map(grp)
months=sorted(train.billing_month.unique())
blocks=[months[i:i+3] for i in range(len(months)-12,len(months),3)]  # last 12 months, 4 folds
folds=[]
for k,b in enumerate(blocks):
    va=np.where((train.g==k)&train.billing_month.isin(b))[0]
    tr=np.where((train.g!=k)&(train.billing_month<b[0]))[0]
    folds.append((tr,va))
    print(k,b[0].date(),len(tr),len(va))
def cv(model):
    oof=[];ys=[];aps=[]
    for tr,va in folds:
        m=clone(model).fit(train.loc[tr,FEATURES],train.loc[tr,TARGET])
        p=m.predict_proba(train.loc[va,FEATURES])[:,1]
        oof.append(p);ys.append(train.loc[va,TARGET].values);aps.append(average_precision_score(ys[-1],p))
    return np.mean(aps),np.std(aps),np.concatenate(ys),np.concatenate(oof)
res={n:cv(m) for n,m in CANDS.items()}
pd.DataFrame({n:{'cv_AP':r[0],'sd':r[1],'cv_AUC':roc_auc_score(r[2],r[3])} for n,r in res.items()}).T.sort_values('cv_AP',ascending=False)

## 4. Choice (CV only) and threshold

In [ ]:
best=max((n for n in res if n!='baseline'),key=lambda n:res[n][0])
y,p=res[best][2],res[best][3]
ths=np.linspace(0.02,0.8,79)
f1s=[f1_score(y,p>=t) for t in ths]
THR=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(THR,3),'| CV F1:',round(max(f1s),3))

### Recorded choice (written before the test set is scored)
The model is the one printed above (highest CV average precision among the non-trivial candidates), using features `tariff, occupants, kwh, bill_eur, late_payments_12m`, with the F1-maximising CV threshold printed above. `final_notice_sent` is excluded as leakage. None of these are revisited after the test score.

## 5. Refit on all training rows; score the test set once

In [ ]:
final=clone(CANDS[best]).fit(train[FEATURES],train[TARGET])
pt=final.predict_proba(test[FEATURES])[:,1]; yt=test[TARGET].values
base=clone(CANDS['baseline']).fit(train[FEATURES],train[TARGET]).predict_proba(test[FEATURES])[:,1]
yh=(pt>=THR).astype(int)
print(f'Test AUC {roc_auc_score(yt,pt):.3f} | AP {average_precision_score(yt,pt):.3f} (baseline AP {average_precision_score(yt,base):.3f})')
print(f'At threshold {THR:.2f}: precision {precision_score(yt,yh):.3f}, recall {recall_score(yt,yh):.3f}, F1 {f1_score(yt,yh):.3f}')

## Conclusion
The test numbers above are the headline: a single evaluation on later months for accounts never seen in training. Caveats: one split; the test set holds only ~275 accounts. Adding `final_notice_sent` would look far stronger but is not available when predicting, so it is excluded deliberately.

Results (this run): chosen model logreg (C=0.1), threshold 0.06. Test AUC 0.73, AP 0.47 vs 0.22 for the prevalence baseline; at the threshold precision 0.33, recall 0.70. So there is real but modest signal.
Important caveat: arrears prevalence rises from 5.8% in training to 21.7% in the test months, so the CV-chosen threshold (0.06) flags many accounts and precision is low. The pooled CV AUC for the baseline (0.69) is an artefact of pooling folds with different prevalences, not skill. The notebook outputs were produced by running the cells in a plain Python process because the Jupyter kernel could not start in the sandbox.